[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-03-output-parsers.ipynb#scrollTo=a3b4c5d6)

---
# Day 3 · Output Parsers and Structured Outputs
**certified-journeys / llm-engineering-certified** · Day 3 · Parsing

> **Goal for today:** By the end of this notebook you can extract clean strings with `StrOutputParser`, define Pydantic models and parse structured output with `PydanticOutputParser`, recover from malformed output using `OutputFixingParser`, and extract schema-free JSON with `JsonOutputParser`.


In [ ]:
%pip install -q langchain langchain-openai langchain-community pydantic python-dotenv


## Step 1 · Setup and why output parsers matter

A raw `ChatOpenAI` call returns an `AIMessage` object. Your application usually needs one of:
- A **plain string** (for display, downstream text processing)
- A **typed Python object** (for database writes, API calls)
- A **dict / JSON blob** (for flexible schema-free extraction)

Output parsers sit at the end of a chain and convert `AIMessage → your target type`. They also provide `get_format_instructions()` — a prompt snippet that tells the LLM what format to emit.

| Parser | Output type | Use when |
|---|---|---|
| `StrOutputParser` | `str` | You just want the text |
| `PydanticOutputParser` | Pydantic model instance | You need typed, validated fields |
| `JsonOutputParser` | `dict` | Schema is flexible or unknown at write time |
| `OutputFixingParser` | Same as inner parser | LLM output is malformed and needs auto-repair |

**Official docs:** https://python.langchain.com/docs/concepts/output_parsers/


In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-replace-with-real-key"
    print("[INFO] Using placeholder API key — set OPENAI_API_KEY in your .env file.")
else:
    print("[OK] OPENAI_API_KEY loaded from environment.")


### What just happened?
- Same environment setup as Day 2 — `.env` or a Colab Secret for `OPENAI_API_KEY`.
- We install `pydantic` explicitly because `PydanticOutputParser` uses it for schema generation.
- All parser demos below work with mocked responses so you can explore without a real API key.


## Step 2 · `StrOutputParser` — extract clean string output

`StrOutputParser` is the simplest parser: it unwraps `.content` from an `AIMessage` and returns a plain Python string.

When to use it:
- Displaying text in a UI
- Passing output to another string-processing step
- Streaming — `StrOutputParser` works seamlessly with `.stream()`

Chain pattern: `prompt | llm | StrOutputParser()`


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=256)

# Build a simple summarisation chain.
summary_template = ChatPromptTemplate.from_messages([
    ("system", "Summarise the following text in exactly one sentence."),
    ("human", "{text}"),
])

# Add StrOutputParser at the end of the chain.
str_chain = summary_template | llm | StrOutputParser()

# Demonstrate the difference: without vs with the parser.
raw_response = AIMessage(
    content="LangChain is an open-source framework that provides a unified interface for building applications powered by large language models.",
    usage_metadata={"input_tokens": 40, "output_tokens": 24, "total_tokens": 64},
    response_metadata={"model_name": "gpt-4o-mini", "finish_reason": "stop"},
)

# Without parser: you get an AIMessage object.
print("Type WITHOUT parser:", type(raw_response).__name__)
print("Value:", raw_response)  # full AIMessage repr
print()

# With StrOutputParser: you get a plain str.
parser = StrOutputParser()
clean_string = parser.invoke(raw_response)  # parser accepts AIMessage directly
print("Type WITH StrOutputParser:", type(clean_string).__name__)
print("Value:", clean_string)


### What just happened?
- Without the parser, `.invoke()` returns an `AIMessage` — a LangChain object, not a string.
- `StrOutputParser().invoke(ai_message)` extracts `.content` and returns a plain `str`.
- **In a chain** (`prompt | llm | StrOutputParser()`), the output of the whole chain is a `str` — no `.content` attribute needed downstream.
- `StrOutputParser` is also the correct parser to use with `.stream()` — it yields string chunks directly.


## Step 3 · `PydanticOutputParser` — typed, validated extraction

`PydanticOutputParser` wraps a Pydantic model class. It does two things:
1. **`get_format_instructions()`** — generates a JSON schema description you include in your prompt so the LLM knows what to emit.
2. **`.parse(text)`** — validates and deserialises the LLM's JSON output into a typed Python object.

Always include `format_instructions` in your prompt — without it, the LLM emits whatever format it prefers, and parsing will fail unpredictably.

**Official docs:** https://python.langchain.com/docs/how_to/output_parser_pydantic/


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Define the schema we want the LLM to fill.
class BookReview(BaseModel):
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author's full name")
    rating: int = Field(description="Rating from 1 to 5", ge=1, le=5)
    summary: str = Field(description="One-sentence summary of the book")
    recommended: bool = Field(description="Whether you recommend this book")

# Create the parser.
pydantic_parser = PydanticOutputParser(pydantic_object=BookReview)

# IMPORTANT: always embed format_instructions in the prompt.
format_instructions = pydantic_parser.get_format_instructions()
print("Format instructions (first 400 chars):")
print(format_instructions[:400])
print("...")
print(f"\nTotal chars in format instructions: {len(format_instructions)}")


### What just happened?
- Pydantic `Field(description=...)` is how the parser generates human-readable schema instructions — always add a description.
- `get_format_instructions()` produces a JSON Schema + prose explanation that you inject into your prompt.
- **The instructions are ~300–500 characters** — this token cost is fixed regardless of input size, so it's worth it for reliable parsing.
- Validators like `ge=1, le=5` are enforced at parse time — malformed output raises `ValidationError` (handled in Step 4).


In [ ]:
# Build the prompt with format_instructions injected via a {format_instructions} variable.
book_template = ChatPromptTemplate.from_messages([
    ("system", "You are a book review assistant. Extract structured information from the review text.\n\n{format_instructions}"),
    ("human", "{review_text}"),
])

# LCEL chain: template → llm → pydantic parser.
book_chain = book_template | llm | pydantic_parser

# --- MOCK BLOCK: simulate a valid LLM JSON response ---
mock_json_response = """```json
{"title": "The Pragmatic Programmer", "author": "David Thomas and Andrew Hunt", "rating": 5, "summary": "A timeless guide to software craftsmanship, covering everything from personal responsibility to career development.", "recommended": true}
```"""

# PydanticOutputParser.parse() strips markdown fences and validates.
parsed_review: BookReview = pydantic_parser.parse(mock_json_response)
# --- end mock block ---

# Uncomment for live call:
# review_text = "The Pragmatic Programmer by Thomas & Hunt is a must-read. Five stars. Highly recommended for every developer."
# parsed_review = book_chain.invoke({"review_text": review_text, "format_instructions": format_instructions})

print("Parsed BookReview object:")
print(f"  title:       {parsed_review.title}")
print(f"  author:      {parsed_review.author}")
print(f"  rating:      {parsed_review.rating}/5")
print(f"  recommended: {parsed_review.recommended}")
print(f"  summary:     {parsed_review.summary}")
print()
print("Python type:", type(parsed_review).__name__)
print("JSON export:", parsed_review.model_dump_json())


### What just happened?
- `pydantic_parser.parse()` strips markdown code fences, parses JSON, and runs Pydantic validation — all in one step.
- The result is a **real Python object** with IDE autocomplete, type hints, and `.model_dump_json()` for serialisation.
- Field constraints (`ge=1, le=5`) are enforced — a `rating: 7` from the LLM would raise `ValidationError` immediately.
- `.model_dump_json()` converts back to JSON — useful for storing results in a database or sending to an API.


## Step 4 · `OutputFixingParser` — auto-repair malformed output

Even with format instructions, LLMs occasionally produce malformed JSON — missing quotes, trailing commas, extra prose. `OutputFixingParser` wraps any inner parser and, on failure, sends the bad output back to the LLM with a repair prompt.

When to use it:
- Production pipelines where a parse failure would break the app
- Working with models that are less reliable at JSON (e.g., smaller or fine-tuned models)

When NOT to use it:
- If the model consistently produces bad output — fix the prompt first
- When budget is tight — `OutputFixingParser` doubles API calls on failure

**Official docs:** https://python.langchain.com/docs/how_to/output_parser_fixing/


In [ ]:
from langchain.output_parsers import OutputFixingParser

# Create the fixing parser: wraps pydantic_parser + uses llm to repair.
fixing_parser = OutputFixingParser.from_llm(parser=pydantic_parser, llm=llm)

# Simulate a malformed LLM response — common failure modes:
malformed_responses = [
    # Missing closing brace
    '{"title": "Clean Code", "author": "Robert Martin", "rating": 4, "summary": "Best practices for readable code.", "recommended": true',
    # Extra prose before JSON
    'Here is the extracted info: {"title": "Clean Code", "author": "Robert Martin", "rating": 4, "summary": "Best practices.", "recommended": true}',
    # Wrong type (rating as string instead of int)
    '{"title": "Clean Code", "author": "Robert Martin", "rating": "four", "summary": "Best practices.", "recommended": true}',
]

for i, bad_output in enumerate(malformed_responses, 1):
    print(f"--- Malformed response {i} ---")
    print("Input:", bad_output[:80], "..." if len(bad_output) > 80 else "")

    # First, try the inner parser directly — this will fail.
    try:
        result = pydantic_parser.parse(bad_output)
        print("Direct parse: SUCCESS (unexpected)")
    except Exception as e:
        print(f"Direct parse: FAILED — {type(e).__name__}: {str(e)[:60]}")

    print()


### What just happened?
- All three malformed outputs fail the inner `PydanticOutputParser` — as expected.
- `OutputFixingParser` would catch these failures and send a repair prompt to the LLM.
- **Case 3** (wrong type) is especially important: `pydantic` coerces `"four"` → raises `ValidationError`, not silently converts.
- The fixing parser adds one extra LLM call on failure — budget ~2× your normal per-call cost when `OutputFixingParser` is in the chain.


In [ ]:
# Demonstrate OutputFixingParser recovering from a fixable error.
# In mock mode we simulate what the fixing parser's LLM repair call would return.

# The extra prose case is easiest to repair — the LLM just strips the prefix.
fixable_output = 'Here is the extracted info: {"title": "Clean Code", "author": "Robert Martin", "rating": 4, "summary": "Best practices for readable code.", "recommended": true}'

# Simulate the repaired output the fixing parser's LLM call would produce.
repaired_json = '{"title": "Clean Code", "author": "Robert Martin", "rating": 4, "summary": "Best practices for readable code.", "recommended": true}'

# Parse the repaired output through the inner parser.
fixed_review = pydantic_parser.parse(repaired_json)

# In a live pipeline, replace the two lines above with:
# fixed_review = fixing_parser.parse(fixable_output)  # automatically calls LLM to repair

print("Recovered from malformed output:")
print(f"  title:  {fixed_review.title}")
print(f"  author: {fixed_review.author}")
print(f"  rating: {fixed_review.rating}")
print()
print("Strategy: OutputFixingParser sends the bad output + error message to the LLM")
print("asking it to produce a corrected version. This adds one extra API call on failure.")


### What just happened?
- `OutputFixingParser.parse(bad_text)` first tries the inner parser; on failure it calls the LLM with a correction prompt.
- The correction prompt includes the original bad output AND the error message — giving the LLM enough context to fix it.
- **This is a last resort** — if your LLM consistently produces malformed output, improve your format instructions first.
- Consider logging when the fixing parser is invoked — a high fix rate signals a prompt quality issue.


## Step 5 · `JsonOutputParser` — schema-free JSON extraction

`JsonOutputParser` extracts JSON without a predefined Pydantic schema. Use it when:
- The JSON structure varies per call (e.g., extracting entities from unstructured text)
- You want a `dict` and will validate or transform it yourself downstream
- You're prototyping and the schema isn't finalised yet

It still supports `get_format_instructions()` — always include it so the LLM knows to emit JSON.


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser with no schema — returns raw dict.
json_parser = JsonOutputParser()

# Build a prompt for entity extraction.
entity_template = ChatPromptTemplate.from_messages([
    ("system",
     "Extract all named entities from the text. Return a JSON object with keys: "
     "'people' (list of names), 'organisations' (list), 'locations' (list), "
     "'dates' (list).\n\n{format_instructions}"),
    ("human", "{text}"),
])

entity_chain = entity_template | llm | json_parser

# Format instructions tell the LLM to emit valid JSON.
json_format_instructions = json_parser.get_format_instructions()
print("JsonOutputParser format instructions:")
print(json_format_instructions)


### What just happened?
- `JsonOutputParser()` with no arguments accepts any valid JSON — no schema enforcement.
- `get_format_instructions()` returns a short directive telling the LLM to return a JSON markdown code block.
- **Always inject format instructions** — without them, the LLM may return prose, partial JSON, or Markdown tables.
- If you need the output validated, use `PydanticOutputParser` instead; use `JsonOutputParser` only when schema flexibility is intentional.


In [ ]:
# Simulate an entity extraction call and verify the result with json.loads().

sample_text = (
    "On 15 March 2024, Sundar Pichai announced that Google DeepMind would open a new "
    "research centre in London. The event was attended by UK Prime Minister Rishi Sunak "
    "at 10 Downing Street."
)

# --- MOCK BLOCK: simulate the LLM JSON response ---
mock_entity_json = """```json
{"people": ["Sundar Pichai", "Rishi Sunak"], "organisations": ["Google DeepMind"], "locations": ["London", "10 Downing Street", "UK"], "dates": ["15 March 2024"]}
```"""

# JsonOutputParser strips fences and parses to dict.
entities: dict = json_parser.parse(mock_entity_json)
# --- end mock block ---

# Uncomment for live call:
# entities = entity_chain.invoke({"text": sample_text, "format_instructions": json_format_instructions})

# Verify: re-encode with json.loads(json.dumps(...)) to confirm it's valid JSON.
round_tripped = json.loads(json.dumps(entities))

print("Extracted entities:")
for key, values in entities.items():
    print(f"  {key:15s}: {values}")

print()
print("json.loads(json.dumps(result)) == result:", round_tripped == entities)
print("Python type of result:", type(entities).__name__)


### What just happened?
- `json_parser.parse()` strips the markdown code fence and returns a Python `dict` — no schema required.
- **`json.loads(json.dumps(result))`** is a zero-dependency round-trip check: if this succeeds, the data is JSON-serialisable and structurally valid.
- Accessing `entities["people"]` works immediately — no `.content` unwrapping, no manual `json.loads()` call.
- For downstream processing, you can use `entities.get("dates", [])` with a safe default — the LLM may omit keys if the text contains no examples.


## Step 6 · Putting it all together — a parser selection guide

The right parser depends on what you need downstream:

```
Do you need a Python object with validation?  ──→  PydanticOutputParser
Do you need a dict, schema unknown at write time? ──→  JsonOutputParser
Do you just need the text?  ──→  StrOutputParser
Is your parser failing in production? ──→  wrap it in OutputFixingParser
```

Let's build a router that picks the right parser per task.


In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define a second Pydantic model for a different use case.
class BugReport(BaseModel):
    severity: str = Field(description="One of: critical, high, medium, low")
    component: str = Field(description="The affected system component")
    description: str = Field(description="One-sentence bug description")
    reproduction_steps: list[str] = Field(description="Ordered list of reproduction steps")

bug_parser = PydanticOutputParser(pydantic_object=BugReport)

# Parser registry: task_type → (parser, format_instructions_key)
PARSERS = {
    "summarise":    (StrOutputParser(),  None),
    "extract_json": (JsonOutputParser(), "format_instructions"),
    "file_bug":     (bug_parser,         "format_instructions"),
}

def get_chain(task_type: str, system_prompt: str):
    """Build a chain for the given task type using the appropriate parser."""
    parser, fi_key = PARSERS[task_type]
    if fi_key:
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt + "\n\n{format_instructions}"),
            ("human", "{input}"),
        ])
    else:
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
        ])
    return prompt | llm | parser

# Verify chain types.
for task in PARSERS:
    chain = get_chain(task, "You are a helpful assistant.")
    # The last element in the chain is the parser — check its type.
    print(f"{task:<20} → parser: {type(chain.last).__name__}")


### What just happened?
- A **parser registry** lets you select the right parser at runtime without duplicating chain logic.
- `chain.last` is the final step in the LCEL chain — useful for introspection and testing.
- `format_instructions` is conditionally injected only when the parser needs it — `StrOutputParser` doesn't use it.
- This pattern scales to a microservice where different endpoints use different output schemas.


In [ ]:
# Exercise the BugReport parser with a mock incident description.

bug_chain = get_chain(
    "file_bug",
    "You are a QA engineer. Extract a structured bug report from the incident description.",
)

# --- MOCK BLOCK ---
mock_bug_json = """```json
{"severity": "critical", "component": "authentication", "description": "Users are logged out immediately after login when 2FA is enabled.", "reproduction_steps": ["Navigate to /login", "Enter valid credentials", "Enable 2FA and enter OTP", "Observe redirect to /dashboard followed by immediate redirect back to /login"]}
```"""
bug_report: BugReport = bug_parser.parse(mock_bug_json)
# --- end mock block ---

# Uncomment for live call:
# incident = "After enabling 2FA, users are kicked back to login immediately. Affects all auth flows. Happening since the 14:30 deploy."
# bug_report = bug_chain.invoke({"input": incident, "format_instructions": bug_parser.get_format_instructions()})

print(f"Severity:    {bug_report.severity.upper()}")
print(f"Component:   {bug_report.component}")
print(f"Description: {bug_report.description}")
print(f"Steps:")
for i, step in enumerate(bug_report.reproduction_steps, 1):
    print(f"  {i}. {step}")
print()
print("Is a BugReport instance:", isinstance(bug_report, BugReport))


### What just happened?
- `reproduction_steps: list[str]` is a **typed list** — Pydantic validates that each element is a string and that the field is present.
- `isinstance(bug_report, BugReport)` confirms we have a real Python object, not a dict — IDE autocomplete works on it.
- In production, you'd call `bug_report.model_dump()` to get a dict for database insertion, or `bug_report.model_dump_json()` to send to a webhook.
- The same `BugReport` schema can be reused across multiple chains — define once, use anywhere.


In [ ]:
# Challenge: End-to-end pipeline with error handling
# ─────────────────────────────────────────────────
# Build a pipeline that:
#   1. Defines a Pydantic model `ProductListing` with fields:
#      - name: str
#      - price_usd: float
#      - in_stock: bool
#      - tags: list[str]
#
#   2. Creates a PydanticOutputParser for ProductListing
#
#   3. Builds a ChatPromptTemplate that:
#      - Has a system message instructing the model to extract product info
#      - Injects format_instructions
#      - Has a human slot {product_description}
#
#   4. Wraps the parser in an OutputFixingParser
#
#   5. Parses these three inputs (use mocked LLM responses if no API key):
#      a. "Blue wireless headphones, $49.99, currently available, great for music and calls"
#      b. "Premium coffee maker - SOLD OUT - $129 - tags: kitchen, appliance, premium"
#      c. (simulate a malformed response to test OutputFixingParser)
#
#   6. Print a summary table: name | price | in_stock | tags

# Your solution here
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers import OutputFixingParser

# TODO: define ProductListing
# TODO: create parser and fixing_parser
# TODO: build template with format_instructions
# TODO: parse all three inputs and print table


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `StrOutputParser` | Extracts `.content` from `AIMessage` → plain `str`; use at end of display/streaming chains |
| `PydanticOutputParser` | Validates and deserialises JSON → typed Python object; always inject `get_format_instructions()` |
| `get_format_instructions()` | Without it, the LLM doesn't know what format to emit — always include it in the prompt |
| `OutputFixingParser` | Wraps any parser; on failure sends bad output + error back to LLM for repair — costs 2× on failure |
| `JsonOutputParser` | Schema-free `dict` extraction; flexible but no validation — good for prototyping |
| `json.loads(json.dumps(x))` | Fast round-trip check that a Python dict is fully JSON-serialisable |
| Parser registry pattern | Map task types to parsers at module level; avoids duplicating chain logic |

> **Tip:** Always include the parser's `get_format_instructions()` in your prompt — without it the LLM doesn't know what format to emit.

---
## What's next
**Day 4** → LCEL Chains — compose complex multi-step pipelines using LangChain Expression Language, branching logic, and parallel execution.

Mark Day 3 complete in your [tracker](../index.html).
